In [17]:
import pandas as pd
import numpy as np
from pandas import DataFrame, Series
import joblib

import matplotlib.pyplot as plt
import seaborn as sb

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

### Kaggle login to download the dataset

In [3]:
import kagglehub
import shutil
import pathlib

# Download latest version
dataset_path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")

print("Path to dataset files:", dataset_path)

# copy files from that path to current working directory
path = pathlib.Path(dataset_path)
destination = pathlib.Path.cwd() / "data"
destination.mkdir(parents=True, exist_ok=True)

for file in path.iterdir():
    if file.is_file(): shutil.copy2(file, destination)

C:\Users\liyak\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\liyak\.cache\kagglehub\datasets\janiobachmann\bank-marketing-dataset\versions\1


In [4]:
data = pd.read_csv(pathlib.Path("data/bank.csv"))
data

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


In [5]:
data.describe()

,age,balance,day,duration,campaign,pdays,previous
count,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000
mean,41.231948,1528.538524,15.658036,371.993818,2.508421,51.330407,0.832557
std,11.913369,3225.413326,8.420740,347.128386,2.722077,108.758282,2.292007
min,18.000000,-6847.000000,1.000000,2.000000,1.000000,-1.000000,0.000000
25%,32.000000,122.000000,8.000000,138.000000,1.000000,-1.000000,0.000000
50%,39.000000,550.000000,15.000000,255.000000,2.000000,-1.000000,0.000000
75%,49.000000,1708.000000,22.000000,496.000000,3.000000,20.750000,1.000000
max,95.000000,81204.000000,31.000000,3881.000000,63.000000,854.000000,58.000000


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


In [7]:
categorical_columns = data.select_dtypes(include=['object']).columns.to_list()
numerical_columns = data.select_dtypes(include=['int64', 'float64']).columns.to_list()

categorical_columns.remove('deposit')

print(categorical_columns)
print(numerical_columns)

['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [8]:
numeric_pipeline = Pipeline([
    ('standard_scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('categorical_pipeline', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('numerical_conversion', numeric_pipeline, numerical_columns),
    ('categorical_columns', categorical_pipeline, categorical_columns)
])

In [9]:
X, y = data.drop('deposit', axis=1), data['deposit']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Save the test data in the current working directory

In [11]:
test_data = X_test.copy()
test_data['deposit'] = y_test

test_data.to_csv('test_data.csv', index=False)

### Define the models with some hyperparameter values and then fit them into the pipeline

In [18]:
models = {
    "logistic_regression": LogisticRegression(C=2),
    "decision_tree": DecisionTreeClassifier(ccp_alpha=2),
    "knn": KNeighborsClassifier(n_jobs=5),
    "naive_bayes": GaussianNB(),
    "svc": SVC(kernel='rbf')
}

models_path = pathlib.Path('models')
models_path.mkdir(parents=True, exist_ok=True)

results = []

positive_label = "yes"
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        classes = list(pipeline.classes_)
        positive_index = classes.index(positive_label)
        y_scores = pipeline.predict_proba(X_test)[:, positive_index]
    elif hasattr(pipeline, "decision_function"):
        classes = list(pipeline.classes_)
        y_scores = pipeline.decision_function(X_test)
        if classes[1] != positive_label:
            y_scores = -y_scores
    else:
        raise TypeError(f"{name} cannot provide scores for AUC.")

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(
            (y_test == positive_label).astype(int),
            y_scores,
        ),
        "Precision": precision_score(
            y_test, y_pred, pos_label=positive_label, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, pos_label=positive_label, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred, pos_label=positive_label, zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    joblib.dump(pipeline, models_path / f"{name}.pkl")

results_df = (
    pd.DataFrame(results)
    .set_index("Model")
    .sort_values("F1", ascending=False)
)

display(results_df.round(4))

,Accuracy,AUC,Precision,Recall,F1,MCC
Model,,,,,,
svc,0.8554,0.9177,0.8244,0.8828,0.8526,0.7125
knn,0.8298,0.8848,0.8291,0.8072,0.8180,0.6585
logistic_regression,0.8253,0.9072,0.8275,0.7977,0.8123,0.6495
naive_bayes,0.7179,0.8100,0.8048,0.5340,0.6420,0.4489
decision_tree,0.5262,0.5000,0.0000,0.0000,0.0000,0.0000


In [13]:
results_df = DataFrame(results)
results_df = results_df.sort_values(by='Accuracy', ascending=False)
results_df

,Model,Accuracy
4,svc,0.855352
2,knn,0.829825
0,logistic_regression,0.825347
3,naive_bayes,0.717868
1,decision_tree,0.526198


In [14]:
test_data = pd.read_csv("test_data.csv")

# Separate features and target
X_test = test_data.drop(columns=["deposit"])
y_test = test_data["deposit"]

# Select one sample
sample = X_test.iloc[[0]]

# Actual target value
actual_value = y_test.iloc[0]

print("Sample Customer Details:")
display(sample)

print("Actual Subscription:", actual_value)

Sample Customer Details:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,32,management,married,tertiary,no,393,no,no,cellular,28,jan,458,2,-1,0,unknown


Actual Subscription: no


In [15]:
models = {
    "Logistic Regression": joblib.load("models/logistic_regression.pkl"),
    "Decision Tree": joblib.load("models/decision_tree.pkl"),
    "KNN": joblib.load("models/knn.pkl"),
    "Gaussian Naive Bayes": joblib.load("models/naive_bayes.pkl"),
    "SVM": joblib.load("models/svc.pkl")
}

In [16]:
prediction_results = []

for model_name, model in models.items():

    prediction = model.predict(sample)[0]

    prediction_results.append({
        "Model": model_name,
        "Actual Value": actual_value,
        "Predicted Value": prediction,
        "Correct Prediction": prediction == actual_value
    })

prediction_df = pd.DataFrame(prediction_results)

display(prediction_df)

,Model,Actual Value,Predicted Value,Correct Prediction
0,Logistic Regression,no,yes,False
1,Decision Tree,no,no,True
2,KNN,no,no,True
3,Gaussian Naive Bayes,no,no,True
4,SVM,no,yes,False
